**A Time ordered difference is the difference between a current row’s value and the previous (or next) row’s value based on chronological order such as date, timestamp, month, or sequence**

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,TimestampType,DateType

In [0]:
schema=StructType(fields=[StructField("id",IntegerType(),nullable=True),
                          StructField("event_date",DateType(),nullable=True)])

In [0]:
from datetime import date

data = [
    (101, date(2026, 1, 20)),
    (101, date(2026, 1, 25)),
    (102, date(2026, 1, 23)),
    (102, date(2026, 1, 26)),
    (103, date(2026, 1, 23)),
    (103, date(2026, 1, 24)),
    (103, date(2026, 1, 25)),
    (104, date(2026, 1, 25)),
    (105, date(2026, 1, 26)),
    (105, date(2026, 1, 25)),
    (106, date(2000, 4, 16))
]

In [0]:
df=spark.createDataFrame(data,schema)
df.createOrReplaceTempView("df")

In [0]:
display(df)

In [0]:
df_final=spark.sql(f"""select id,event_date,lag(event_date) over(partition by id  order by event_date) as prev_time 
                   ,event_date-lag(event_date) over(partition by id order by event_date ) as time_diff  from df""")
display(df_final)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
window_spec=Window.partitionBy("id").orderBy("event_date")

In [0]:
df_2=df.withColumn("prev_event_date",F.lag(df.event_date).over(window_spec))
display(df_2)

In [0]:
df_spark_final_v2=df_spark_final.withColumn("time_dff",df_spark_final.event_date-df_spark_final.prev_event_date)
display(df_spark_final_v2)